# create new coal df

In [1]:
import numpy as np
import re

from carbon_bombs.io.gem import load_coal_mine_gem_database


def _year_match(x):
    match = re.search(r'\b\d{4}\b', str(x))

    if match:
        return int(match.group())

    return 3000

def _add_rank_columns(df_gem):    
    
    df_gem["year"] = df_gem["start_year"].apply(
        lambda x: _year_match(x)
    )
    df_gem["rank"] = df_gem.groupby("project_name").year.rank(method="first", ascending=True)



def load_gem_database():
        
    df_gem = load_coal_mine_gem_database()
    
    # renaming to match CB
    manual_matching = {
        "Troyanovo 3 Coal Mine": "Maritsa Coal Mines",
        "Troyanovo 1 Coal Mine": "Maritsa Coal Mines",
        "Troyanovo-North Coal Mine": "Maritsa Coal Mines",
        "Kuu-Check Coal Mine": "Borly Coal Mines",
        "Molodezhny Coal Mine": "Borly Coal Mines",
        "Kolubara D & E Coal Mine": "Kolubara Mine Complex",
        "Kolubara G Coal Mine": "Kolubara Mine Complex",
        "Radljevo Coal Mine": "Kolubara Mine Complex",
        "Tamnava-West Coal Mine": "Kolubara Mine Complex",
        "Shubarkol Centralny Coal Mine": "Shubarkol Coal Mine",
        "Shubarkol Zapadny Coal Mine": "Shubarkol Coal Mine",
        "East Well of Sihe Coal Mine": "Sihe Coal Mine",
        "West Well of Sihe Coal Mine": "Sihe Coal Mine",
        "Wilton Coal Mine": "Wilton and Fairhill Coal Projects",
        "Fairhill Coal Mine": "Wilton and Fairhill Coal Projects",
    }
    df_gem["Mine Name"] = df_gem["Mine Name"].replace(manual_matching)

    # map columns
    GEM_cols_mapping = {
        "GEM Mine ID": "gem_id",
        "Mine Name": "project_name",
        "Country / Area": "country",
        "Latitude": "latitude",
        "Longitude": "longitude",
        "Parent Company": "parent_company",
        "Status": "project_status",
        "Project Type": 'project_type',
        "Project Phase": "project_phase",
        "Opening Year": "start_year",
        "Total Reserves (Proven and Probable, Mt)": "reserves",
        "Total Resource (Inferred, Indicated, Measured)": "resources",
        "Coal Type": "coal_type",
        "Coal Grade": "coal_grade",

        'Closing Year': 'closing_year', 
        'Year of Total Reserves Recorded': 'recorded_year', 
        "GEM_url": "gem_url",
    }

    df_gem["GEM_url"] = np.where(
        df_gem["GEM Wiki Page (ENG)"].isna(), df_gem["GEM Wiki Page (Non-ENG)"], df_gem["GEM Wiki Page (ENG)"]
    )

    # keep only wanted columns
    df_gem = df_gem.loc[:, GEM_cols_mapping.keys()]
    df_gem = df_gem.rename(columns=GEM_cols_mapping)

    # put project status to lowercase
    df_gem["project_status"] = df_gem["project_status"].str.lower()
    
    # update country name
    df_gem["country"] = df_gem["country"].replace({
        "Türkiye": "Turkey"
    })

    # format reserves and resources columns
    df_gem["reserves"] = df_gem["reserves"].replace('-', np.nan) 
    df_gem["resources"] = df_gem["resources"].replace('-', np.nan) 

    # Retrieve reserves (to estimate emission) based on the paper
    # "Coal reserves are defined in this dataset as “recoverable reserves”[...] "
    # " When recoverable reserve figures were unavailable, we collected data on coal resources and indicated those in the dataset in Reserve category."
    df_gem["volume_million_tons"] = np.where(df_gem["reserves"].isna(), df_gem["resources"], df_gem["reserves"])
    
    df_gem["reserve_category_name"] = np.where(
        ~df_gem["reserves"].isna(), "Reserve", np.where(
             ~df_gem["resources"].isna(), "Resource", None
        )
    )

    # Consider 'Bituminous (Met)' type
    df_gem["coal_type"] = np.where(
        (df_gem["coal_type"] == 'Bituminous') & (df_gem["coal_grade"].str.contains("Met")),
        'Bituminous (Met)',
        df_gem["coal_type"]
    )

    # here the dataframe has multiple rows for the same Mine since we can have some expension
    
    # KEEP 1 ROW for each mine, use the one with the first year
    # special cases e.g. Angus Place Coal Mine --> no opening year for both bt Expansion took as first
    _add_rank_columns(df_gem)

    # Coal emission factors based on the paper
    coal_type_factor = {
        'Bituminous': 0.00244068,
        'Bituminous (Met)': 0.00266772,
        'Subbituminous': 0.00181629,
        'Lignite': 0.0012019,
        'Anthracite': 0.00262461,
        'Default': 0.002
    }

    # See what to do for these cases
    # 'Anthracite&Bituminous', 'Subbituminous / Lignite', 'Bituminous and Subbituminous'
    # TODO
    df_gem["coal_type"] = df_gem["coal_type"].replace(
        {
            'Anthracite&Bituminous': 'Anthracite',
            'Subbituminous / Lignite': 'Subbituminous',
            'Bituminous and Subbituminous': 'Bituminous',
        }
    ).fillna('Default')
    
    # Compute emissions
    df_gem["emissions_factor"] = df_gem["coal_type"].apply(
        lambda x: (coal_type_factor[x] if x in coal_type_factor else None)
    )
    df_gem["emissions"] = (
        df_gem["volume_million_tons"] * df_gem["emissions_factor"] * np.where(df_gem["reserve_category_name"] == 'Resource', 0.488, 1)
    )

    # keep only one row per mine
    df_final = df_gem.loc[df_gem["rank"] == 1]

    assert len(df_final) == df_gem["project_name"].nunique()
    assert set(df_final["project_name"]) == set(df_gem["project_name"])

    # ========== TEMP FOR ANALYSIS ========== #
    df_gem["calculation_detail"] = np.where(
        df_gem["volume_million_tons"].fillna(0) == 0,
        None,
        "( "
        + df_gem["volume_million_tons"].astype(str).fillna("")
        + ' ['
        + df_gem["coal_type"].fillna("") 
        + " - " 
        + df_gem["reserve_category_name"].fillna("none")
        + np.where(
            df_gem["reserve_category_name"] == 'Resource', ' (apply 48.8% ratio)', ''
        )
        + "] * "
        + df_gem["emissions_factor"].round(6).astype(str)
        + " ) + "
    )

    # Compute total emissions
    total_em_df = df_gem.groupby("project_name").agg(
        total_emissions=("emissions", "sum"),
        total_volume=("volume_million_tons", "sum"),
        calculation_detail=("calculation_detail", "sum"),
    )

    # merge to add total emissions to the final df
    df_final = df_final.merge(
        total_em_df,
        on="project_name"
    )

    # Final post process
    df_final["total_emissions"] = df_final["total_emissions"].astype(float).round(3)
    df_final["year"] = df_final["year"].replace(3000, None)
    df_final["calculation_detail"] = df_final["calculation_detail"].str[:-3]

    # hardcoded fix for some mines with double entered volume (for operating and expansion project)
    mines_to_fix = [
        'Elk Creek Mine (WV)',
        'Guanyinshan No. 1 Coal Mine',
        'Guizhou Xingyi Xingfa Coal Mine',
        'Huale Coal Mine',
        'Ketki Coal Mine',
        'Mayixi No.1 Coal Mine',
        'Ngaka Coal Mine',
        'North Urimari Birsa Coal Mine',
        'Novoshakhtinskoye Coal Mine',
        'Shitoumei No.1 Coal Mine',
        'Tevshiin Govi Coal Mine',
        'West Well of Faer Second Coal Mine',
        'Western Coal Project',
        'Xinyuan UG Coal Mine'
    ]
    cols_to_fix = [
        "total_emissions", "total_volume"
    ]

    df_final.loc[df_final["project_name"].isin(mines_to_fix), cols_to_fix] = (
        df_final.loc[df_final["project_name"].isin(mines_to_fix), cols_to_fix] / 2
    )
    df_final.loc[df_final["project_name"].isin(mines_to_fix), "calculation_detail"] = (
        df_final.loc[df_final["project_name"].isin(mines_to_fix), "calculation_detail"] + ' divided by 2 (error in GEM source)'
    )
    
    # Sanity check to be sure that we only have one row per mine
    assert len(df_final) == df_gem["project_name"].nunique()
    assert set(df_final["project_name"]) == set(df_gem["project_name"])

    return df_final


In [2]:
import pandas as pd
from carbon_bombs.conf import FPATH_SRC_KHUNE_PAPER


def load_paper_database():
    
    df_paper = pd.read_excel(
        FPATH_SRC_KHUNE_PAPER, sheet_name="Coal", engine="openpyxl", skipfooter=3
    )
    
    cols_paper = {
        'Project Name': 'project_name',
        'Country': 'country',
        'Potential emissions (GtCO2)': 'total_emissions',
        'Status': 'project_status',
        'Reserves (Million tons)': 'volume_million_tons',
        'Reserve category name': 'reserve_category_name',
        'Emissions factor': 'emissions_factor',
        'Coal type': 'coal_type',
        'Year': 'start_year',
        "Source": "source_paper"
        # 'New': 'New'
    }
    
    df_paper = df_paper[cols_paper.keys()]
    df_paper = df_paper.rename(columns=cols_paper)
    
    manual_matching = {
        "Afşin-Elbistan Coal Mine": "Afşin-Elbistan Coal Mines",
        "Bernice-Cygnus Coal Mine": "Berenice-Cygnus Coal Mine",
        "Bogatyr Coal Mine": "Bogatyr Coal Mine (Kazakhstan)",
        "Changcheng No.3 Coal Mine": "Changcheng No. 3 Coal Mine",
        "Dananhu No. 7 Coal Mine": "Dananhu No. 7 Mine",
        "Dananhu No.1 Coal Mine": "Dananhu No. 1 Coal Mine",
        "Dananhu West No.2 Coal Mine": "Dananhu No. 2 Surface Mine",
        "Hongshaquan No.1 Coal Mine": "Hongshaquan No. 1 Surface Mine",
        "Huangling No.2 Coal Mine": "Huangling No. 2 Coal Mine",
        "Jiangjun Gebi No.2 Coal Mine": "Jiangjun Gebi No. 2 Coal Mine",
        "Kaniha Coal Mine": "Gopalji Kaniha Coal Mine",
        "Project Motheo": "Morupule Coal Mine",
        "Shahaiji No.1 Coal Mine": "Shajihai No.1 Coal Mine",
        "Shajihai No.2 Coal Mine": "Xinjiang Shajihai Coal Mine",
        "Talike District No. 2 Coal Mine": "Talike No. 2 Coal Mine",
        "Ulug-Khem Project": "Ulug-Khem Coal Mine",
        "West Macedonia Lignite Centre (WMLC)": "West Macedonia Lignite Centre",
        "Xinwen Ili No.1 Coal Mine": "Xinwen IlI No.1 Coal Mine",
        "Yallourn": "Yallourn Coal Mine",
        "Yangchangwan No.1 Well Coal Mine": "Yangchangwan Coal Mine",
        "Yangquan No.1 Coal Mine": "Yangquan No. 1 Coal Mine",
        "Yimin Surface Coal Mine": "Yimin Surface Mine",
        "Banhardih": "Banhardih Coal Mine",
        "Fording River": "Fording River Operations",
        "Gare Pelma Sector II": "Gare Palma II Coal Mine",
        "Hamilton County Mine No.1": "Hamilton Mine No. 1",
        "Kerandari BC": "Kerendari Coal Mine",
        "Mandakini B": "Mandakini–B Coal Mine",
        "Saharpur Jamarpani": "Saharpur-Jamarpani Coal Mine",
        "Thar Coal Mine": "Thar Block II Coal Mine",
        "Sengwe Colliery": "Sengwa Coal Mine",
        # "Alpha North Coal Mine": "Galilee Coal Mine", --> Galilee Coal Mine already exists
        "BIB Coal Mine": "Borneo Indobara Coal Mine"
    }
    
    df_paper["project_name"] = df_paper["project_name"].replace(
        manual_matching
    )
    
    df_paper["country"] = df_paper["country"].replace({
        "Russian Federation": "Russia"
    })
    
    df_paper["total_emissions_adjusted"] = np.where(
        df_paper["reserve_category_name"] == "Resource", df_paper["total_emissions"] * 0.488, df_paper["total_emissions"]
    )
    
    return df_paper

In [3]:
df_final = load_gem_database()
df_paper = load_paper_database()

/var/folders/r6/k1_4rtzs0v5fcfz8hm016hmm0000gn/T/ipykernel_39745/2028154953.py:87: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_gem["reserves"] = df_gem["reserves"].replace('-', np.nan)
/var/folders/r6/k1_4rtzs0v5fcfz8hm016hmm0000gn/T/ipykernel_39745/2028154953.py:88: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_gem["resources"] = df_gem["resources"].replace('-', np.nan)


In [4]:
# df_paper

In [5]:
from carbon_bombs.utils.location import get_world_region

df_paper_ = df_paper[["project_name", "country", "total_emissions_adjusted", "start_year", "project_status"]].copy()

df_coal_final = df_final.merge(
    df_paper_,
    on=["project_name", "country"],
    how="outer",
    suffixes=('', '_paper')
)

in_paper = ~df_coal_final["total_emissions_adjusted"].isnull()
in_gem = ~df_coal_final["gem_id"].isnull()
gem_over_1gt = df_coal_final["total_emissions"] >= 1
status_to_ignore = df_coal_final["project_status"].isin(["mothballed", "shelved", "cancelled"])

df_coal_final["start_year"] = pd.to_numeric(df_coal_final["start_year"],  errors='coerce')

df_coal_final["project_evolution"] = np.where(
    (in_paper) & (in_gem) & (gem_over_1gt) & (~status_to_ignore), "carbon_bomb_above_1gt", np.where(
        (~in_paper) & (in_gem) & (gem_over_1gt) & (~status_to_ignore), "new_identified_carbon_bomb", np.where(
            (in_paper) & (in_gem) & ((~gem_over_1gt) | (status_to_ignore)), "carbon_bomb_now_below_1gt", np.where(
                (in_paper), 'carbon_bomb_not_found', np.where(
                    (in_gem) & (df_coal_final["start_year"] >= 2021) & (~status_to_ignore), 'new_extraction_projects', 'none'
                )
            )
        )
    )
)

df_coal_final = df_coal_final.loc[
    df_coal_final["project_evolution"] != 'none'
]

# df_coal_final["total_emissions"] = np.where(df_coal_final["total_emissions"].isnull(), df_coal_final["total_emissions_paper"], df_coal_final["total_emissions"])
df_coal_final["start_year"] = np.where(df_coal_final["start_year"].isnull(), df_coal_final["start_year_paper"], df_coal_final["start_year"])
# df_coal_final["project_status"] = np.where(df_coal_final["project_status"].isnull(), df_coal_final["project_status_paper"], df_coal_final["project_status"])
# df_coal_final["project_status"] = df_coal_final["project_status"].str.lower()

df_coal_final["project_status"] = df_coal_final["project_status"].replace({
    "operating": "operating",
    "mothballed": "operating",
    "not started": "not started",
    "in development": "not started",
    "proposed": "not started",
    "discovered": "not started",
    "shelved": "not started",
    "cancelled": "stopped",
    "shut in": "stopped"
})

world_region_df = df_coal_final["country"].drop_duplicates().to_frame().reset_index(drop=True)
world_region_df["world_region"] = world_region_df["country"].apply(get_world_region)

df_coal_final = df_coal_final.merge(world_region_df, on='country')

# remove new field with 0.05 emissions or below
df_coal_final = df_coal_final.loc[
    ~(
        (
            df_coal_final["project_evolution"] == "new_extraction_projects"
        ) & (
            df_coal_final["total_emissions"] <= 0.05
        )
    )
]

# to map gasoil methodology, we set the total emission for not found CB to 0
df_coal_final.loc[df_coal_final["project_evolution"] == "carbon_bomb_not_found", "total_emissions"] = 0

df_coal_final["project_type"] = np.where(df_coal_final["project_evolution"] == 'new_extraction_projects', 'New mines', 'Carbon bombs')
df_coal_final["fuel_type"] = 'Coal'
df_coal_final["new_extraction_after_2021"] = (df_coal_final["start_year"] > 2021).astype(str)

In [6]:
df_coal_final["project_evolution"].value_counts()

project_evolution
new_extraction_projects       237
carbon_bomb_above_1gt         164
new_identified_carbon_bomb    147
carbon_bomb_now_below_1gt      61
carbon_bomb_not_found           5
Name: count, dtype: int64

In [7]:
df_coal_final["project_name"].nunique(), df_coal_final.shape

(614, (614, 33))

In [8]:
not_found_mines = (
    df_coal_final.loc[df_coal_final["project_evolution"] == 'carbon_bomb_not_found']
)[["project_name", "country", "total_emissions"]]

not_found_mines

,project_name,country,total_emissions
7,Alpha North Coal Mine,Australia,0.0
31,Bankui,India,0.0
130,Elga Coal Mine,Russia,0.0
315,Inaglinskaya-2 Mine,Russia,0.0
419,Listvianskaya Coal Mine,Russia,0.0


In [9]:
def extract_name_and_percentage(s):
    s = s.strip()
    match = re.match(r'^(.*?)\s*\[(\d+(?:\.\d+)?)%\]$', s)

    if not s.endswith('%]'):
        return (s, 0)

    if match:
        name = match.group(1).strip()
        percentage = float(match.group(2)) / 100
        percentage = 0 if percentage is None else percentage
        return (name, percentage)
        
    return None

def format_companies(x):
    res = []

    # no companies
    if x is np.nan:
        return res

    # sanity check
    if not isinstance(x, list):
        print(x)
        raise

    # share indicated
    if any([v.endswith('%]') for v in x]):
        comp_with_share = [extract_name_and_percentage(value) for value in x]
        total_prct = sum([val[1] for val in comp_with_share])

        # adjust percentage or add others
        if round(total_prct, 1) != 1:
            if total_prct > 1:
                comp_with_share = [(val[0].strip(), val[1] / total_prct) for val in comp_with_share]
            else:
                comp_with_share = comp_with_share + [('Others', 1 - total_prct)]
        
        return comp_with_share

    # share but no share indicated
    elif len(x) > 1:
        return [(value.strip(), 1 / len(x)) for value in x]

    return [(x[0], 1.)]


companies_with_share = (
    df_coal_final
    .set_index('project_name')["parent_company"]
    .str
    .strip()
    .str
    .replace('%] ', '%];')
    .str
    .replace('[ ', '[')
    .str
    .replace('%0]', '%]')
    .str
    .replace('%）', '%]')
    .str
    .replace('%]I', '%]')
    .str
    .replace('%].', '%]')
    .str
    .replace('%]N', '%];N')
    .str
    .replace('%]，', '%];')
    .str
    .replace(' 49%]', ' [49%]')
    .str
    .replace('[[', '[')
    .str
    .split(';')
    .apply(format_companies)
)

companies_with_share = companies_with_share.explode().reset_index()
companies_with_share[["company", "involvement_interval"]] = pd.DataFrame(companies_with_share["parent_company"].tolist())
companies_with_share = companies_with_share.drop(columns='parent_company')

coal_bocc_gem_companies_matching_df = pd.read_csv("../data_sources/coal_companies_bocc_gem_matching.csv")
bocc_gem_comp_matching = (
    coal_bocc_gem_companies_matching_df[["Parent_Company GEM", "BOCC_company_name"]]
    .set_index("Parent_Company GEM")["BOCC_company_name"]
    .dropna()
    .to_dict()
)

companies_with_share["company"] = companies_with_share["company"].replace(bocc_gem_comp_matching)
companies_with_share["involvement_interval"] = companies_with_share["involvement_interval"].fillna(0).apply(lambda x: str(int(x * 100)) + "%")

companies_with_share = companies_with_share.merge(
    df_coal_final[["project_name", "project_type", "fuel_type"]],
    on="project_name",
    how="left"
)
companies_with_share = companies_with_share.rename(columns={"project_name": "project_name_raw"})

companies_with_share.head()

,project_name_raw,company,involvement_interval,project_type,fuel_type
0,Aduunchuluun Coal Mine,Mongolyn Alt Corporation [MAK],100%,Carbon bombs,Coal
1,Afşin-Elbistan Coal Mines,Euas Electricity Generation Company (Elektrik ...,50%,Carbon bombs,Coal
2,Afşin-Elbistan Coal Mines,Celikler Seyitomer Elektrik Uretim A.S.,50%,Carbon bombs,Coal
3,Alexander Coal Project,Sasol,100%,New mines,Coal
4,Alpha North Coal Mine,NaN,0%,Carbon bombs,Coal


In [10]:
companies_with_share_ = companies_with_share.copy()
companies_with_share_ = companies_with_share_.rename(columns={"project_name_raw": "project_name"})

companies_with_share_ = companies_with_share_.loc[~companies_with_share_.company.isna()]
companies_with_share_["text"] = companies_with_share_["company"] + " (" + companies_with_share_["involvement_interval"] + ")"

companies_with_share_ = companies_with_share_.groupby("project_name").agg(
    list_of_company_involved=("text", lambda x: " | ".join(x))
).reset_index()

In [11]:
columns_to_keep = {
    'project_name': 'project_name',
    'country': 'country',
    'latitude': 'latitude',
    'longitude': 'longitude',
    'world_region': 'world_region',
    'start_year': 'start_year',
    'total_emissions_adjusted': 'total_potential_emissions_v1',
    "total_emissions": "total_potential_emissions",
    # 'producing_potential_emissions': 'producing_potential_emissions',
    # 'short_term_expansion_potential_emissions': 'short_term_expansion_potential_emissions',
    # 'long_term_expansion_potential_emissions': 'long_term_expansion_potential_emissions',
    'project_evolution': 'project_evolution',
    'project_type': 'project_type',
    'new_extraction_after_2021': 'new_extraction_after_2021',
    'project_status': 'project_status',
    'list_of_company_involved': 'list_of_company_involved',
    'fuel_type': 'fuel_type',
    # '2020_2025_past_emissions': '2020_2025_past_emissions',
}


df_coal_final_ = df_coal_final.loc[df_coal_final["project_evolution"] != 'none']
df_coal_final_ = df_coal_final_.merge(
    companies_with_share_,
    on="project_name",
    how="left"
)
# df_coal_final_["producing_potential_emissions"] = ''
# df_coal_final_["short_term_expansion_potential_emissions"] = ''
# df_coal_final_["long_term_expansion_potential_emissions"] = ''


df_coal_final_ = df_coal_final_[columns_to_keep.keys()]
df_coal_final_ = df_coal_final_.rename(columns=columns_to_keep)

# df_coal_final_.to_excel("coal_carbon_bombs_dataset_WIP.xlsx", index=False)

In [12]:
# df_coal_final_.project_evolution.value_counts()

In [13]:
# df_coal_final_.head()

In [14]:
with pd.ExcelWriter("coal_output_data.xlsx") as writer:  
    df_coal_final_.to_excel(writer, sheet_name="coal_project_data", index=False)
    companies_with_share.to_excel(writer, sheet_name="project_companies", index=False)

In [17]:
df_coal_final_.to_csv("coal_project_data.csv", index=False)